# 01 — Event register: load, inspect, verify

Goal of this notebook:

1. Load the seeded policy event register
2. Inspect what's in it (counts, date coverage, confidence)
3. Identify events that need verification before the rest of the framework can rely on them
4. Provide a structured workflow for verifying each one

This notebook should be re-run periodically as the register grows.

In [8]:
import sys
from pathlib import Path

# Make the src package importable
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
from nem_herding import (
    load_events,
    events_in_window,
    events_for_rez,
    events_of_category,
    events_of_coupling_layer,
    register_summary,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

## Load and summarise

In [9]:
events = load_events('../data/events/policy_events.csv')
print(f'Loaded {len(events)} events')
print(f'Date range: {events["date"].min()} to {events["date"].max()}')
print(f'Partial-date events: {events["date_is_partial"].sum()}')
print(f'Verified events: {(events["confidence"] == "verified").sum()}')
register_summary(events)

Loaded 42 events
Date range: 2020-07-30 to 2025-07-15
Partial-date events: 28
Verified events: 10


,field,value,count
0,category,commitment,18
1,category,information,13
2,category,competitive,6
3,category,adverse,5
4,jurisdiction,NSW,19
5,jurisdiction,QLD,7
6,jurisdiction,AEMO,6
7,jurisdiction,FED,6
8,jurisdiction,AEMC,2
9,jurisdiction,FED/NSW/SA,1


## What needs verifying

Events with `confidence == 'needs_verify'` or partial dates (`XX` in the source CSV).
Work through these systematically — fix the date, replace the source URL with a primary source,
set confidence to `verified`.

In [10]:
needs_work = events[
    (events['confidence'] == 'needs_verify') | events['date_is_partial']
][['event_id', 'date', 'date_is_partial', 'jurisdiction', 'event_type', 'description', 'source_url']]
needs_work

,event_id,date,date_is_partial,jurisdiction,event_type,description,source_url
2,e003,2020-11-30,False,NSW,rez_designation,Central-West Orana REZ declared (first NSW REZ),https://www.energyco.nsw.gov.au/renewable-energy-zones/c...
4,e037,2021-07-15,True,NSW,mlf_reset,Annual MLF reset published by AEMO,https://aemo.com.au
5,e004,2021-08-15,True,NSW,rez_designation,New England REZ declared,https://www.energyco.nsw.gov.au/renewable-energy-zones/n...
6,e005,2021-12-15,True,NSW,rez_designation,South West NSW REZ declared,https://www.energyco.nsw.gov.au/renewable-energy-zones/s...
7,e007,2022-02-15,True,NSW,rez_designation,Illawarra offshore wind zone designated,
9,e006,2022-07-15,True,NSW,rez_designation,Hunter-Central Coast REZ declared,https://www.energyco.nsw.gov.au/renewable-energy-zones/h...
10,e036,2022-07-15,True,NSW,mlf_reset,Annual MLF reset published by AEMO,https://aemo.com.au
11,e025,2022-07-15,True,FED/NSW/SA,transmission_milestone,EnergyConnect (PEC) construction milestones,https://www.transgrid.com.au
12,e019,2022-07-22,False,AEMC,rule_change,Transmission Access Reform - directions paper,https://www.aemc.gov.au
13,e008,2022-08-19,False,NSW,eoi_round,Central-West Orana access rights EOI round opens,https://www.energyco.nsw.gov.au


## Events by REZ

In [11]:
# How many events affect each REZ (including state-wide and NEM-wide events that affect all)?
rezs = ['CWO', 'NER', 'SWN', 'HCC', 'ILW', 'QLD_NORTH', 'QLD_CENTRAL', 'QLD_SOUTH']
for rez in rezs:
    rez_events = events_for_rez(events, rez)
    rez_specific = rez_events[rez_events['rez'] == rez]
    print(f'{rez:15s}  total relevant: {len(rez_events):3d}  '
          f'rez-specific: {len(rez_specific):3d}')

CWO              total relevant:  29  rez-specific:   6
NER              total relevant:  26  rez-specific:   3
SWN              total relevant:  24  rez-specific:   1
HCC              total relevant:  24  rez-specific:   1
ILW              total relevant:  24  rez-specific:   1
QLD_NORTH        total relevant:  24  rez-specific:   1
QLD_CENTRAL      total relevant:  24  rez-specific:   1
QLD_SOUTH        total relevant:  24  rez-specific:   1


## Events by coupling layer

The pre-registered prediction is that Layer 2 (informational) events should produce larger
synchronisation responses in mature REZs, while Layer 1 (direct) events dominate in early-stage
REZs. Sanity check: do we have a reasonable mix of both layers in the register?

In [12]:
events.groupby(['coupling_layer', 'category']).size().unstack(fill_value=0)

category,adverse,commitment,competitive,information
coupling_layer,,,,
both,1,18,1,6
direct,0,0,5,1
informational,4,0,0,6


## Verification workflow per event

Pick an event_id from the `needs_work` table above. For each:

1. Open the source_url (or search if it's blank)
2. Confirm exact date
3. Edit `data/events/policy_events.csv` directly — set date, source_url, confidence='verified'
4. Reload here and confirm it's no longer in `needs_work`

Suggested batching: do *all NSW REZ designations* in one session, then *all federal CIS events*
in another, then *all AEMO publications*, etc. Within-type batching reuses your context and
is much faster than jumping around.

In [13]:
# Example: show one event in detail
event_id = 'e004'  # New England REZ declaration - needs date verification
events[events['event_id'] == event_id].T

,5
event_id,e004
date,2021-08-15
jurisdiction,NSW
event_type,rez_designation
category,commitment
coupling_layer,both
information_unification,3
scope,rez
rez,NER
description,New England REZ declared
